# Building an Agent from Scratch with Groq

An agent is a language model that can take actions and work towards a goal over several steps, not just answer in one shot. Before using a framework that hides the machinery, this notebook builds a small agent by hand with the configured chat model (Groq by default), so you can see exactly what an agent loop is and where its memory comes from.

## Learning Objectives

At the end of this notebook, you should be able to:

- Explain why a language model call is stateless and what that means for memory.
- Maintain a conversation by hand by appending each turn to a messages list.
- Describe tools to the model with JSON schemas and parse the tool calls it returns.
- Build the agentic loop: run the requested tools, feed the results back, and repeat until a final answer.

## Setup

We keep the agent loop manual and share only the model connection. `llm_config.py` reads the selected provider and matching API key from `.env`; choose Groq, Mistral, OpenRouter or Google AI Studio. Every example uses the same model at `temperature=0` for repeatable output.

In [ ]:
import json
from collections.abc import Callable

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage

from llm_config import create_chat_model, response_text

In [ ]:
load_dotenv(".env")

model = create_chat_model()

## Language models are stateless

A chat completion call is a single request and response. The model does not remember anything between calls: each request must carry everything the model needs to know. The two calls below are independent, so the second one cannot recall what the first one was told.

In [ ]:
first = model.invoke(
    [{"role": "user", "content": "My name is Ada. Please remember it."}]
)
response_text(first)

In [ ]:
second = model.invoke([{"role": "user", "content": "What is my name?"}])
response_text(second)

The second call has no idea who Ada is. It replies that it does not know your name and explains that each interaction starts fresh. Nothing carried over, because we sent a brand new `messages` list containing only the second question. This is the fact to build on: memory is not something the model has, it is something we provide.

## Giving the model memory

If the model cannot remember, we remember for it. We keep a running `messages` list and append every turn: the user's message, then the model's reply, then the next user message, and so on. Each call sends the whole history, so the model can see what came before.

In [ ]:
messages: list[BaseMessage | dict[str, object]] = [
    {"role": "user", "content": "My name is Ada. Please remember it."}
]

# First reply, added back into the conversation.
reply = model.invoke(messages)
assistant_content = response_text(reply)
messages.append({"role": "assistant", "content": assistant_content})

# Ask the follow-up on the same, growing messages list.
messages.append({"role": "user", "content": "What is my name?"})
reply = model.invoke(messages)
response_text(reply)

This time the model answers that your name is Ada. Nothing changed about the model: the difference is that we sent the earlier turns along with the new question. Carrying the `messages` list forward is the whole mechanism behind an agent's short-term memory.

## Adding tools

So far the model can only produce text. A tool lets it ask for an action: a calculation, a lookup, an API call. The important part is that the model does not run anything itself. It returns a request that names a tool and its arguments, and our code runs the matching Python function and sends the result back.

We describe each tool to the model with a JSON schema (its name, what it does, and its arguments). The loop then works like this:

```mermaid
flowchart TD
    U["Add the user message to messages"] --> M["Call the model with messages and tools"]
    M --> D{"Did the model request tools?"}
    D -->|yes| T["Run each tool, append its result to messages"]
    T --> M
    D -->|no| F["Final answer"]
```

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


def get_change_window(service: str) -> str:
    """Return the approved change window for a service."""
    windows = {"billing-api": "22:00 UTC", "search-api": "18:00 UTC"}
    return windows.get(service, "Window not found")


# Map each tool name to the Python function that runs it.
tool_functions: dict[str, Callable[..., object]] = {
    "multiply": multiply,
    "get_change_window": get_change_window,
}


def parse_tool_arguments(
    raw_arguments: str | dict[str, object] | None,
) -> dict[str, object]:
    """Parse a tool-call JSON object and reject other JSON values."""
    arguments = (
        json.loads(raw_arguments or "{}")
        if isinstance(raw_arguments, str)
        else (raw_arguments or {})
    )
    if not isinstance(arguments, dict):
        raise TypeError("Tool arguments must be a JSON object.")
    return arguments


# Describe the same tools to the model as JSON schemas.
tools: list[dict[str, object]] = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two integers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_change_window",
            "description": "Return the approved change window for a service.",
            "parameters": {
                "type": "object",
                "properties": {"service": {"type": "string"}},
                "required": ["service"],
            },
        },
    },
]

## The agentic loop

Now we run the loop. We send the conversation and the tool schemas, then inspect the reply. If the model asked for tools, we run each one, append its result to `messages` as a `tool` message, and call the model again. When the model stops asking for tools, it has produced its final answer and we stop. We cap the loop at eight model calls and send malformed tool arguments back as errors.

In [ ]:
messages: list[BaseMessage | dict[str, object]] = [
    {
        "role": "user",
        "content": "Multiply 6 by 7, and tell me the approved change window for billing-api.",
    }
]
final_answer = ""

for _ in range(8):
    response = model.bind_tools(tools).invoke(messages)
    message = response

    # No tool calls means the model has produced its final answer.
    if not message.tool_calls:
        if not message.text.strip():
            raise RuntimeError("The provider returned no final answer.")
        final_answer = response_text(message)
        messages.append({"role": "assistant", "content": final_answer})
        break

    # Keep the assistant turn that requested the tools, then run each one.
    messages.append(message)
    for call in message.tool_calls:
        try:
            arguments = parse_tool_arguments(call["args"])
            function = tool_functions.get(call["name"])
            if function is None:
                raise ValueError(f"Unknown tool: {call['name']}")
            result = function(**arguments)
        except (TypeError, ValueError) as exc:
            result = f"Tool error: {exc}"
        messages.append(
            {"role": "tool", "tool_call_id": call["id"], "content": str(result)}
        )

else:
    raise RuntimeError("Agent did not finish within 8 steps.")

final_answer

A typical run calls the model twice: first it requests both tools, so our code runs `multiply(6, 7)` (42) and `get_change_window("billing-api")` (22:00 UTC). It then uses those results to answer. The model may request tools in separate turns, so inspect your actual trace rather than assuming a fixed number of calls.

## Inspect the context we built

The whole conversation now lives in the `messages` list that we assembled by hand. Reading it back shows the full trace of the run.

In [ ]:
def describe(message: BaseMessage | dict[str, object]) -> str:
    if isinstance(message, BaseMessage):
        role, content = message.type, message.text
        tool_calls = getattr(message, "tool_calls", [])
    else:
        role = str(message["role"])
        content = str(message.get("content") or "")
        tool_calls = message.get("tool_calls") or []
    if tool_calls:
        content = "requests tools: " + ", ".join(call["name"] for call in tool_calls)
    return f"{role:9} | {content[:80]}"


for index, message in enumerate(messages):
    print(index, describe(message))

If both tools were requested in one turn, the context has five messages: the user question, the assistant's tool requests, two `tool` results, and the final answer. Separate tool requests add more turns. That list is the agent's memory of the run, and we built it ourselves.

## Summary

In this notebook you:

- Saw that a language model call is stateless: it remembers nothing between requests.
- Gave the model memory by hand by carrying a growing `messages` list.
- Described tools to the model with JSON schemas and parsed the tool calls it returned.
- Built the agentic loop: run the requested tools, append the results, and repeat until a final answer.

This hand-written loop is the foundation for everything that follows. In the next notebook you apply it yourself to build a SQL agent over a real database. Later, LangChain's `create_agent(...)` runs this same loop for you, so you can build agents without writing the plumbing each time.

## References & Further Reading

- [**Groq Quickstart**](https://console.groq.com/docs/quickstart): Make your first chat completion with the configured chat model (Groq by default).
- [**Groq Text Generation**](https://console.groq.com/docs/text-chat): The Chat Completions API and the messages format.
- [**Groq Tool Use**](https://console.groq.com/docs/tool-use): Function calling and the tool-call loop.
- [**Groq Console**](https://console.groq.com/playground): Create a free API key and try the model used here.